# Road Traffic Accidents in Finland 2015–2024
## Weather, season and accident severity — a CRISP-DM analysis

**Course:** Datan käsittely ja koneoppiminen · **Process model:** CRISP-DM

| Phase | Section |
|---|---|
| 1 | Business Understanding |
| 2 | Data Understanding (accidents + FMI weather) |
| 3 | Data Preparation |
| 4 | Modeling |
| 5 | Evaluation |
| 6 | Deployment |

**Data sources**
- **Road traffic accidents:** Statistics Finland (Tilastokeskus) WFS service, layers `tieliikenne_2015 … tieliikenne_2024`. One row = one accident that led to injury or death, with year, month, hour, severity, type, involved vehicle groups and EPSG:3067 coordinates.
- **Weather:** Finnish Meteorological Institute (FMI) open data, daily observations from all stations in Finland.

**Folder layout expected by this notebook**
```
traffic_accidents_analysis.ipynb
datasets/
    tieliikenne_2015.csv ... tieliikenne_2024.csv
    fmi_cache/            <- created automatically (one small CSV per downloaded month)
```

## 1. Business Understanding

### 1.1 Background and objective
Road safety work needs to know **when** accident risk is high and **what makes an accident severe**. A common belief is that winter conditions (cold, snow, ice) are the main trigger of severe accidents. We test this belief with data instead of assuming it.

An important limitation shapes the whole project: the accident data contain **year and month but no exact date**. Every link between accidents and weather therefore has to be made at **monthly resolution**.

### 1.2 Research questions
- **RQ1:** Can the severity of an injury accident (fatal or serious injury vs. lighter injury) be predicted from the characteristics of the accident (accident type, road users involved, time of day)?
- **RQ2:** Do environmental and seasonal variables (monthly mean temperature, precipitation, snow depth and share of frost days from the nearest weather station) improve the prediction, and can they predict severity on their own?

### 1.3 Hypotheses
- **H1:** Months with harsher winter weather (cold, snow) have a higher share of severe accidents.
- **H2:** Adding weather variables to the accident characteristics improves the prediction of severity.

### 1.4 Success criteria
- **RQ1:** the model performs clearly better than chance, meaning ROC-AUC clearly above 0.5 and PR-AUC clearly above the base rate of severe accidents (about 14%). A model that always answers "less severe" is not a success even if its accuracy looks high, so the severe class must really be detected.
- **RQ2:** weather is considered useful if the model with weather has a higher ROC-AUC and PR-AUC than the same model without weather, evaluated on the same test accidents.

### 1.5 Constraints and assumptions
- Monthly resolution only; weather is a **national monthly aggregate** (not the weather at the accident site).
- Only accidents that led to injury or death are in the data, so we model *injury accidents*, not all collisions.
- No traffic-volume (exposure) data are available; the number of accidents depends on how much people drive.

## 2. Data Understanding


### 2.1 Setup


The project uses data from two sources. The road traffic accident data from **Statistics Finland** is provided as yearly CSV files covering the period from 2015 to 2024. The available files are loaded from the local datasets directory and combined into a single DataFrame.


The weather data is retrieved from the **Finnish Meteorological Institute (FMI) Open Data service** using the stored query. Daily observations on the weather are collected from the weather stations across the Finland. The analysis focuses on daily mean temperatures, precipitation and snow depth.

To make the data collection more efficient, the downloaded **FMI data** is stored locally under the **datasets/fmi_cache/**. If a file is already available, is it loaded from the cache instead of being downloaded again.

At this stage, **the two datasets** are loaded and examined separately. Their structures, variables, datatypes and missing values are inspected before proceeding to data preparation and integration.

In [ ]:
# 1. Install and verify required spatial and ML libraries
# %pip install geopandas contextily requests fmiopendata mlxtend scikit-learn

# 2. Import core engines
import glob
import os
import time
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import contextily as ctx
import datetime as dt

from fmiopendata.wfs import download_stored_query

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, PoissonRegressor
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (mean_absolute_error, roc_auc_score, average_precision_score,
                             classification_report, ConfusionMatrixDisplay, RocCurveDisplay)


from weather_api_functions import _find_value, _download_range, fetch_fmi_month, load_weather_month

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 40)

RANDOM_STATE = 69
DATA_DIR = "datasets"            # folder that holds tieliikenne_YYYY.csv
YEARS = list(range(2015, 2025))
#TEST_YEARS = [2023, 2024]        # final hold-out: the two most recent years (time-based split)

### 2.2 Accident data (Statistics Finland)
All ten yearly files are loaded and stacked. Columns (identical in every file):

| Column | Meaning |
|---|---|
| `vvonn`, `kkonn` | year and month of the accident (no day) |
| `kello` | one-hour interval, format `HH.00-HH.59` |
| `vakav` | severity: 1 = fatal, 2 = injury, 3 = serious injury |
| `onntyyppi` | accident type 0–9 (same/opposite/crossing directions, pedestrian, run-off-road, other) |
| `lkmhapa … lkmmuukulk` | number of involved road users per group (cars and vans, buses and trucks, pedestrians, bicycles, mopeds, motorcycles, others) |
| `x`, `y` | location, EPSG:3067 (ETRS-TM35FIN) |

In [ ]:
files = sorted(glob.glob(os.path.join(DATA_DIR, "tieliikenne_20*.csv")))
print("Files found:", [os.path.basename(f) for f in files])
assert len(files) == len(YEARS), "Expected one file per year 2015-2024"

df_accidents = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

print("\nShape (rows, columns):", df_accidents.shape)
print("\nData types:")
print(df_accidents.dtypes)
display(df_accidents.head())

In total, there are 17 columns of data in the dataset. 

In [ ]:
print("Missing values")
print(df_accidents.isna().sum()[lambda s: s > 0])
print("\nRows with missing 'kello' per year:")
print(df_accidents[df_accidents["kello"].isna()]["vvonn"].value_counts().sort_index())
print("\nDuplicate FIDs:", df_accidents["FID"].duplicated().sum())

Variable (`kello`) is the only one with missing values, with a total of 16.

In [ ]:
# Class variable codebook
SEVERITY_LABELS = {1: "fatal", 2: "injury", 3: "serious injury"}
ACCIDENT_TYPES = {
    0: "same direction (straight)",
    1: "same direction (turning)",
    2: "opposite directions (straight)",
    3: "opposite directions (turning)",
    4: "crossing directions (straight)",
    5: "crossing directions (turning)",
    6: "pedestrian (crossing)",
    7: "pedestrian (elsewhere)",
    8: "run off the road",
    9: "other",
}

counts = pd.DataFrame({
    "severity": df_accidents["vakav"].map(SEVERITY_LABELS).value_counts(),
})
counts["share_%"] = (counts["severity"] / len(df_accidents) * 100).round(1)
display(counts)

type_counts = df_accidents["onntyyppi"].map(ACCIDENT_TYPES).value_counts()
display(type_counts.to_frame("accidents"))

### 2.3 Accidents over time
Two patterns matter for modeling: a **strong long-term decline** and a **seasonal cycle**.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))

per_year = df_accidents.groupby("vvonn").size()
axes[0].bar(per_year.index, per_year.values, color="steelblue")
axes[0].set_title("Accidents per year")
axes[0].set_xlabel("Year"); axes[0].set_ylabel("Accidents")

heat = df_accidents.groupby(["vvonn", "kkonn"]).size().unstack()
sns.heatmap(heat, annot=True, fmt="d", cmap="YlOrRd", cbar=False, ax=axes[1])
axes[1].set_title("Accidents per year and month")
axes[1].set_xlabel("Month"); axes[1].set_ylabel("Year")

plt.tight_layout(); plt.show()

print(f"Change 2015 -> 2024: {per_year.iloc[0]} -> {per_year.iloc[-1]} "
      f"({(per_year.iloc[-1] / per_year.iloc[0] - 1) * 100:.0f} %)")
print("Month with most accidents (all years):", df_accidents.groupby('kkonn').size().idxmax())
print("Month with fewest accidents (all years):", df_accidents.groupby('kkonn').size().idxmin())

In [ ]:
# Share of severe accidents (fatal or serious injury) by year and by month
is_severe = df_accidents["vakav"].isin([1, 3])
share_year = is_severe.groupby(df_accidents["vvonn"]).mean() * 100
share_month = is_severe.groupby(df_accidents["kkonn"]).mean() * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(share_year.index, share_year.values, marker="o", color="crimson")
axes[0].set_title("Severe accidents, % of all accidents, by year")
axes[0].set_xlabel("Year"); axes[0].set_ylim(0, None)
axes[1].plot(share_month.index, share_month.values, marker="o", color="crimson")
axes[1].set_title("Severe accidents, % of all accidents, by month")
axes[1].set_xlabel("Month"); axes[1].set_ylim(0, None)
plt.tight_layout(); plt.show()

print(f"Overall share of severe accidents: {is_severe.mean() * 100:.1f} %")
print("Lowest month:", share_month.idxmin(), f"({share_month.min():.1f} %) | highest month:",
      share_month.idxmax(), f"({share_month.max():.1f} %)")


### 2.4 Weather data (Finnish Meteorological Institute)
**Source:** FMI open data WFS, stored query `fmi::observations::weather::daily::multipointcoverage` (daily observations from all weather stations in the bounding box of Finland).

**Variables requested**

| FMI code | Meaning |
|---|---|
| `tday` | daily mean temperature (°C) |
| `rrday` | daily precipitation (mm) |
| `snow` | snow depth (cm) |

**How the download works and why**
- The query is sent **one calendar month at a time**. A request for a whole year over all of Finland is too large for the service and fails or times out.
- Each downloaded month is saved to `datasets/fmi_cache/`. If a download is interrupted or a month fails, just **run the cell again**: finished months are read from disk, only missing ones are fetched.
- Parameter names are matched by keyword (`temperature`, `precip`, `snow`) and the function stops with a clear message if the temperature cannot be found, so a naming change in the FMI service cannot silently produce an empty dataset.
- If a full-month request fails, the function automatically retries it as two half-month requests.

In [ ]:
weather_parts, failed_months = [], []
for year in YEARS:
    for month in range(1, 13):
        part = load_weather_month(year, month)
        if part is None:
            failed_months.append(f"{year}-{month:02d}")
        else:
            weather_parts.append(part)

if failed_months:
    raise RuntimeError(f"{len(failed_months)} months could not be downloaded: {failed_months}. "
                       "Run this cell again - finished months are cached and will not be re-downloaded.")

df_weather_daily = pd.concat(weather_parts, ignore_index=True).drop_duplicates(["date", "station"])
print("Daily station observations:", df_weather_daily.shape)
display(df_weather_daily.head())

In [ ]:
# Weather data quality check: restrict to 2015-2024 and verify coverage

# Work on a copy
w = df_weather_daily.copy()

# Drop rows outside of 2015-2024
w['year_check'] = w['date'].dt.year
w = w[w['year_check'].isin(YEARS)]

# Coverage per month: how many distinct days and stations have observations
cover = (w.assign(vuosi=w["date"].dt.year, kuukausi=w["date"].dt.month)
          .groupby(["vuosi", "kuukausi"])
          .agg(days=("date", "nunique"), stations=("station", "nunique")))

print("--- FMI Säädatan Laatutarkastus ---")
print("Months covered:", len(cover), "(odotettu: 120)")
print("Days with data per month  : min", cover["days"].min(), "| max", cover["days"].max())
print("Stations per month        : min", cover["stations"].min(), "| max", cover["stations"].max())
print("\nShare of missing values:")
print(w[["tday", "rrday", "snow"]].isna().mean().round(3))
print("\nValue ranges:")
display(w[["tday", "rrday", "snow"]].describe().T)

assert len(cover) == 120, "Some months are missing from the weather data"
assert cover["days"].min() >= 28, "A month has fewer than 28 days of observations"


**Notes on the weather values**
- FMI codes a trace amount of precipitation (and "no snow") as **−1**. These values are set to 0 in the preparation phase.
- Snow depth is measured only at some stations and mostly in winter. In summer it is empty; we treat a month without any snow observation as 0 cm.
- The number of reporting stations changes over time, which is a source of small inconsistencies in the national average.



The `describe()` method of pandas produces summary statistics for each column of a Weather DataFrame. For numeric columns it reports the number of non-missing values (count), the mean, the standard deviation, the minimum and maximum, and the 25th, 50th (median) and 75th percentiles. With the option `include='all'` enabled, it also summarises non-numeric columns by reporting the number of unique values, the most frequent value and its frequency.


In [ ]:
df_weather_daily.describe(include='all')

In [ ]:
# Weather measurement locations
df_weather_daily

## 3. Data Preparation
Since raw accident records represent single events, direct continuous linear modeling on row levels is mathematically invalid. We perform an **Aggregation step** to compute total accidents grouped by month, establishing a true continuous numeric target variable suitable for Logistic Regression modeling.

### 3.1 Accident Data Preparation

#### 3.1.1 Readable Column Names

In [ ]:
# Readable column names for presentation
readable_col_names =  {
"vvonn" : "vuosi",
"kkonn" : "kuukausi",
"kello" : "kello",
"vakav" : "vakavuus",
"onntyyppi" : "onnettomuustyyppi",
"lkmhapa" : "henkilö- ja pakettiautot",
"lkmlaka" : "linja- ja kuorma-autot",
"lkmjk" : "jalankulkijat",
"lkmpp" : "polkupyörät ja kevyet sähköajoneuvot",
"lkmmo" : "mopot",
"lkmmp" : "moottoripyörät",
"lkmmuukulk" : "muut kulkuneuvot",
"x" : "x",
"y" : "y"
}
df_accidents.rename( columns=readable_col_names, inplace=True)

#### 3.1.2 Converting Class Variables to 1-hot Encoded Columns

In [ ]:
# Convert accident severity to 0 or 1
df_accidents['vakavuus'] = df_accidents['vakavuus'].isin([1, 3]).astype(int)

accident_types_raw = {
0 : "samat ajosuunnat (ajo suoraan)",
1 : "samat ajosuunnat (ajo kääntyen)",
2 : "vastakkaiset ajosuunnat (ajo suoraan)",
3 : "vastakkaiset ajosuunnat (ajo kääntyen)",
4 : "risteävät ajosuunnat (ajo suoraan)",
5 : "risteävät ajosuunnat (ajo kääntyen)",
6 : "jalankulkijaonnettomuus (suojatiellä)",
7 : "jalankulkijaonnettomuus (muualla)",
8 : "tieltä suistuminen",
9 : "muu onnettomuus"
}

# Create columns and assign integer 1 or 0
for type_number, column_name in accident_types_raw.items():
    df_accidents[column_name] = [1 if accident_type == type_number else 0 for accident_type in df_accidents['onnettomuustyyppi']]

# Drop onnettomuustyyppi column
df_accidents.drop('onnettomuustyyppi', axis=1, inplace=True)

#### 3.1.3 Converting Cyclical Time Variable to Components
Time is a cyclical variable that overflows at midnight. To make it compatible with logistic regression, we break it down to sine and cosine components of a 24 hour cycle.

The hours are are in String datatype and in format XX.00-XX.59.

We take the first two characters of each string in column 'kello' and convert that to float.

In [ ]:
df_accidents['tunti'] = df_accidents['kello'].str[:2].astype(float)

# drop the row(s) with missing time
df_accidents = df_accidents.dropna(subset=['tunti'])

df_accidents["sin_tunti"] = np.sin(2 * np.pi * df_accidents["tunti"] / 24)
df_accidents["cos_tunti"] = np.cos(2 * np.pi * df_accidents["tunti"] / 24)

# Drop tunti
df_accidents.drop(columns=['tunti'], inplace=True)

#### 3.1.4 Accident Data Location Formatting

In [ ]:
import geopandas as gpd

# 1. Convert EPSG:3067 coordinates (x, y) directly into a GeoDataFrame
accidents_gdf = gpd.GeoDataFrame(
    df_accidents,
    geometry=gpd.points_from_xy(df_accidents["x"], df_accidents["y"]),
    crs="EPSG:3067"
)

# 2. Re-project the ENTIRE dataset at once to WGS84 (EPSG:4326 - lat/lon)
accidents_gdf = accidents_gdf.to_crs(epsg=4326)

### 3.2 Weather Data Preparation


#### 3.2.1 Daily Station Values to One Value Per Month
1. Clean the codes (−1 = trace → 0).
2. Average over all stations to get one national value **per day**.
3. Aggregate the days into **monthly features**: mean temperature, total precipitation, mean snow depth and the share of days with a mean temperature below 0 °C.

In [ ]:
# 1. Create a copy of the raw daily weather table
d_stations = df_weather_daily.copy()

# Clean up trace amounts and missing codes as we did before
d_stations["rrday"] = d_stations["rrday"].clip(lower=0)
d_stations["snow"] = d_stations["snow"].clip(lower=0)

# Extract year and month fields from the datetime index
d_stations["vuosi"] = d_stations["date"].dt.year
d_stations["kuukausi"] = d_stations["date"].dt.month

# 2. Aggregation Phase: Group by Year, Month, Station Name, and Location coordinates!
# This collapses the 10-minute/daily records into a clean Monthly Station Dataset.
df_weather_monthly_stations = d_stations.groupby(
    ["vuosi", "kuukausi", "station", "location"],
    as_index=False
).agg(
    keski_lampotila=("tday", "mean"),       # Monthly mean temp for THIS specific station
    sade_summa=("rrday", "sum"),            # Monthly total rain for THIS specific station
    lumensyvyys=("snow", "mean"),          # Monthly mean snow depth for THIS specific station
    pakkaspaivat_osuus=("tday", lambda s: (s.dropna() < 0).mean()) # Frost days share for THIS station
)

# Handle missing snow values for the stations safely
df_weather_monthly_stations["lumensyvyys"] = df_weather_monthly_stations["lumensyvyys"].fillna(0)

print("Monthly weather dataset with unique locations constructed successfully!")
print(f"Dataset Shape (Monthly Station Records): {df_weather_monthly_stations.shape}")

# Display the first 14 rows to let Abu Shahab see the station names and coordinates
display(df_weather_monthly_stations.head(14))


Convert daily FMI observations into monthly averages
WHILE KEEPING the unique Station Name and its Geographic Location!


In [ ]:
WX_FEATURES = ["keski_lampotila", "sade_summa", "lumensyvyys", "pakkaspaivat_osuus"]

fig, axes = plt.subplots(2, 2, figsize=(14, 7), sharex=True)
x = pd.to_datetime(dict(year=df_weather_monthly_stations["vuosi"], month=df_weather_monthly_stations["kuukausi"], day=1))
titles = ["Mean temperature (°C)", "Total precipitation (mm, station average)",
          "Mean snow depth (cm)", "Share of days below 0 °C"]
for ax, col, title in zip(axes.ravel(), WX_FEATURES, titles):
    ax.plot(x, df_weather_monthly_stations[col], color="teal")
    ax.set_title(title)
plt.tight_layout(); plt.show()

#### 3.2.2 Weather Data Location Formatting


In [ ]:
# Function for converting strings to coordinates
def location_string_to_lonlat(string):
        string = string.strip("{}")
        lon_lat_dict = dict( item.replace("'","").split(": ") for item in string.split(", ") )
        lon = float(lon_lat_dict["longitude"])
        lat = float(lon_lat_dict["latitude"])
        return lat, lon

df_weather_monthly_stations["x"] = [ location_string_to_lonlat(location_string)[0] for location_string in df_weather_monthly_stations["location"] ]
df_weather_monthly_stations["y"] = [ location_string_to_lonlat(location_string)[1] for location_string in df_weather_monthly_stations["location"] ]

# Put weather data to GeoDataFrame
weather_gdf = gpd.GeoDataFrame(
    df_weather_monthly_stations,
    geometry=gpd.points_from_xy(df_weather_monthly_stations["x"], df_weather_monthly_stations["y"]),
    crs="EPSG:4326"
)
weather_gdf = weather_gdf.to_crs(epsg=4326)

### 3.3 Combining the Datasets Based on Location Data

In [ ]:
from geopandas.tools import sjoin_nearest

# Join dataframes
joined_gdf = sjoin_nearest(left_df=accidents_gdf, right_df=weather_gdf, how="left", lsuffix="accident", rsuffix="weather", distance_col="weather_distance")
joined_gdf.drop_duplicates(subset=["FID"], inplace=True)
joined_gdf

In [ ]:
# drop more coluumns
joined_gdf.drop(columns=[
    'FID', 'id', 'geom', 'vuosi_accident', 'kuukausi_accident', 'kello', 'x_accident', 'y_accident', 'geometry', 'index_weather', 'vuosi_weather',
       'kuukausi_weather', 'station', 'location', 'x_weather', 'y_weather', 'weather_distance'
], inplace=True)


In [ ]:
# raw copy if needed later
joined_gdf_raw = joined_gdf.copy(deep=True)

### 3.4 Separating training and testing data

The data is divided into training (75%) and testing (25%). To guarantee same results across every run, a fixed **random_state** is applied.

In [ ]:
from sklearn.model_selection import train_test_split

y = joined_gdf['vakavuus']
X = joined_gdf.drop(columns= 'vakavuus')

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=69
)

We make X_train_acc and _X_test_acc by dropping the columns fetched from Weather Data to test if having seasonal features included has any effect on the Model's results.

In [ ]:
wx_cols = ['keski_lampotila', 'sade_summa', 'lumensyvyys', 'pakkaspaivat_osuus']

X_train_acc = X_train.drop(columns=wx_cols)
X_test_acc  = X_test.drop(columns=wx_cols)

### 3.4 Feature scaling

To prepare the features for moding, we standardize them using **Scikit-Learn's StandardScaler**. This transforms the features to have a mean of 0 and a standard deviation of 1.

The scaling is done due to Logistic Regression is sensitive to the scale of input features. Features with larger numeric ranges could otherwise dominate the model's coefficients.

Scaler is fitted strictly on the training set **X_train** to learn its mean and variance. These values are then applied to the test set **X_test**, ensuring no information from the testing data leaks into the training process.

In [ ]:
## Scale features

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Fit the scaler on X_train data only to learn its mean and variance
X_train_scaled = scaler.fit_transform(X_train)
# Scale X_test using the exact same mean and variance learned from X_train
X_test_scaled = scaler.transform(X_test)

# Wrap results back into a DataFrame
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_train.columns)

# assert the list sizes, small check.
assert len(X_train_scaled) == len(y_train) and len(X_test_scaled) == len(y_test)

print("Data Preparation Completed:")
print(f"Training set size: {X_train.shape[0]} samples.")
print(f"Testing set size: {X_test.shape[0]} samples.")

print("\n1. Without standardize features train data :")
display(X_train.head(5))

print("\n2. With standardize features train data (for the Logistic regression) :")
display(X_train_scaled.head(5))



### 3.X Scale features (accident columns only, no weather)

In [ ]:
scaler_acc = StandardScaler()

# Fit the scaler on X_train_acc data only to learn its mean and variance
X_train_acc_scaled = scaler_acc.fit_transform(X_train_acc)
# Scale X_test_acc using the exact same mean and variance learned from X_train_acc
X_test_acc_scaled = scaler_acc.transform(X_test_acc)

# Wrap results back into a DataFrame
X_train_acc_scaled = pd.DataFrame(X_train_acc_scaled, columns=X_train_acc.columns)
X_test_acc_scaled = pd.DataFrame(X_test_acc_scaled, columns=X_train_acc.columns)

# assert the list sizes, small check.
assert len(X_train_acc_scaled) == len(y_train) and len(X_test_acc_scaled) == len(y_test)

print("Data Preparation Completed (accident columns only):")
print(f"Training set size: {X_train_acc.shape[0]} samples.")
print(f"Testing set size: {X_test_acc.shape[0]} samples.")

print("\n1. Without standardize features train data :")
display(X_train_acc.head(5))

print("\n2. With standardize features train data (for the Logistic regression) :")
display(X_train_acc_scaled.head(5))

## 4. Modeling

## Modeling with combined data

With the data prepared, we initialize and train a **LogisticRegression** model from Scikit-Learn. The model is trained using the **X_train_scaled** and **y_train** that includes data and columns from Statistics Finland and Finnish Meteorological Institure.


In [ ]:
from sklearn.linear_model import LogisticRegression

# Model initialization and training
model1 = LogisticRegression(random_state=69) # class_weight='balanced' poistettu
model1.fit(X_train_scaled, y_train)

## 4.X Logistic regression: accident features only

In [ ]:
# Model initialization and training
model_acc = LogisticRegression(random_state=69)

model_acc.fit(X_train_acc_scaled, y_train)

### 4. X Alternative Cross Validation
Cross validation instead of split validation

In [ ]:
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import accuracy_score

# cross-validation

y_pred = cross_val_predict(estimator=model1, X=X_train_scaled, y=y_train, cv=10)

cm = confusion_matrix(y_train, y_pred)
accuracy = accuracy_score(y_train, y_pred)

print("Accuracy: %0.2f" % accuracy)
print("Confusion Matrix:\n", cm)

# visualize confusion matrix
plt.matshow(cm)
plt.title('Confusion matrix')
plt.colorbar()
plt.ylabel('True label')
plt.xlabel('Predicted label')
# include counts
for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i, j], ha='center', va='center', color='red', backgroundcolor='white')
plt.show()

## 4.X Alternate Model: Random Forest

We wanted to test the dataset with another Model as the Logistic Regression model results did not meet our expectations.

We set the `Random Forest Model` with the following settings:
- 250 trees,
- Max depth 12,
- at least 5 sample sper leaf,
- a random state as 42

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

rf = RandomForestClassifier(
    n_estimators=250,
    max_depth=12,
    random_state=42,
    min_samples_leaf=5,
)

rf.fit(X_train, y_train)

In [ ]:
y_pred = rf.predict(X_test)

rf.score(X_test, y_test)

print(classification_report(y_test, y_pred))

The model reached an accuracy of 86% on the test set of 9589 accidents.

With the `less severe` class the model performed quite well, with a precision of 87% and a recall of 99%. This means that the model almost always recognized these accidents correctly.

With the `severe` class, the model performed poorly as the recall for this was only 7% meaning it found only 7% of the severe accidents and incorrectly predicted the rest to be less severe. However, when the model predicted an accident to be severe, it was correct slightly more than a half of the time.

# 5. Evaluation

## 5.X Logistic Regression with Weather

In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

# Train accuracy
print("Train accuracy:", model1.score(X_train_scaled, y_train))
# Test accuracy
print("Test accuracy: ", model1.score(X_test_scaled, y_test))

In [ ]:
pred = model1.predict(X_test_scaled)
print(classification_report(y_true=y_test, y_pred=pred))

ConfusionMatrixDisplay.from_predictions(
    y_test, pred,
    display_labels=['Less severe', 'Severe'],
    cmap='Blues'
)
plt.title('Logistic regression: confusion matrix')
plt.show()

In [ ]:
print(y_train.value_counts())
print(y_test.value_counts())
print(df_accidents['vakavuus'].value_counts())

## 5.X Logistic Regression without Weather

In [ ]:
# Train accuracy
print("Train accuracy:", model_acc.score(X_train_acc_scaled, y_train))
# Test accuracy
print("Test accuracy: ", model_acc.score(X_test_acc_scaled, y_test))

In [ ]:
pred_acc = model_acc.predict(X_test_acc_scaled)
print(classification_report(y_true=y_test, y_pred=pred_acc))

ConfusionMatrixDisplay.from_predictions(
    y_test, pred_acc,
    display_labels=['Less severe', 'Severe'],
    cmap='Blues'
)
plt.title('Logistic regression without weather: confusion matrix')
plt.show()

## 5.X Does weather improve the prediction?

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

proba = model1.predict_proba(X_test_scaled)[:, 1]
proba_acc = model_acc.predict_proba(X_test_acc_scaled)[:, 1]

print("With weather:    ROC-AUC", round(roc_auc_score(y_test, proba), 3),
      "| PR-AUC", round(average_precision_score(y_test, proba), 3))
print("Without weather: ROC-AUC", round(roc_auc_score(y_test, proba_acc), 3),
      "| PR-AUC", round(average_precision_score(y_test, proba_acc), 3))
print("Base rate:", round(y_test.mean(), 3))

Based on the results, it is clear that **the weather does not improve the severity prediction.** The results are almost identical.

### 5.X Feature Importance

In [ ]:
from sklearn.inspection import permutation_importance

# Calculate permutation importance using the standardized model and scaled test data
result = permutation_importance(
    model1,
    X_train_scaled,
    y_train,
    n_repeats=10,
    scoring="accuracy",
    random_state=69,
    n_jobs=-1
)

# Sort features by their mean importance score
sorted_idx = result.importances_mean.argsort()

# Plot the results
fig, ax = plt.subplots(figsize=(10, 12))

ax.boxplot(
    result.importances[sorted_idx].T,
    orientation="horizontal",
    tick_labels=X_train_scaled.columns[sorted_idx],
    patch_artist=True
)
ax.set_title("Permutation Feature Importance")


#fig.tight_layout()
plt.show()


### 6. Deployment